# Homework 2 — Tiny negotiation society (local Ollama)

Multiagent Learning | Agentic AI (local edition)

Same setup as Homework 1, but the model runs locally via **Ollama** (`llama3.2`).

1. Baseline Buyer / Seller / Mediator
2. Exercise 1 — second seller with a higher floor
3. Exercise 2 — buyer memory across negotiations

## 1. Setup

Needs Ollama running and model `llama3.2` pulled (see README).

In [17]:
import re

import ollama

MODEL = "llama3.2"

try:
    listed = ollama.list()
    models = listed.models if hasattr(listed, "models") else listed.get("models", [])
    names = []
    for m in models:
        if hasattr(m, "model"):
            names.append(m.model)
        elif hasattr(m, "name"):
            names.append(m.name)
        elif isinstance(m, dict):
            names.append(m.get("model") or m.get("name") or "")
except Exception as e:
    raise RuntimeError(
        "Cannot reach Ollama. Install from https://ollama.com and start the app."
    ) from e

# Accept exact name or tagged variants like llama3.2:latest
if not any(str(n).startswith(MODEL) for n in names):
    raise RuntimeError(
        f"Model '{MODEL}' not found. Run: ollama pull {MODEL}\n"
        f"Installed: {sorted(names) or '(none)'}"
    )

print(f"Ready. Model = {MODEL}")

Ready. Model = llama3.2


## 2. Shared primitives

- **Agent** — role prompt + board view + one model call
- **Orchestrator** — round-robin; stop on `DEAL:` or max turns
- **Board** — shared list of `(speaker, text)`

In [18]:
class Agent:
    """LLM-wrapper agent: system prompt is identity + goal; board is the percept."""

    def __init__(self, name: str, role: str, memory: str | None = None):
        self.name = name
        self.role = role
        self.memory = memory

    def _system(self) -> str:
        parts = [
            f"You are {self.name}. {self.role}",
            "Reply with your next line only, 1-2 sentences.",
        ]
        if self.memory:
            parts.append(
                "Memory of past negotiations (use to bargain smarter; "
                "do not invent deals that are not on the current board):\n"
                + self.memory
            )
        return " ".join(parts[:2]) + ("\n\n" + parts[2] if len(parts) > 2 else "")

    def act(self, board: list[tuple[str, str]]) -> str:
        view = "\n".join(f"{who}: {text}" for who, text in board)
        r = ollama.chat(
            model=MODEL,
            messages=[
                {"role": "system", "content": self._system()},
                {"role": "user", "content": view or "Begin."},
            ],
        )
        content = r["message"]["content"] if isinstance(r, dict) else r.message.content
        return (content or "").strip()


def extract_deal_price(board: list[tuple[str, str]]) -> float | None:
    for _, text in board:
        m = re.search(r"DEAL:\s*([0-9]+(?:\.[0-9]+)?)", text, flags=re.IGNORECASE)
        if m:
            return float(m.group(1))
    return None


def run_negotiation(
    agents: list[Agent],
    max_turns: int = 9,
    verbose: bool = True,
) -> dict:
    board: list[tuple[str, str]] = []
    for turn in range(max_turns):
        speaker = agents[turn % len(agents)]
        line = speaker.act(board)
        board.append((speaker.name, line))
        if verbose:
            print(f"{speaker.name}: {line}")
        if re.search(r"DEAL:\s*", line, flags=re.IGNORECASE):
            break
    price = extract_deal_price(board)
    return {
        "board": board,
        "turns": len(board),
        "deal_price": price,
        "deal": price is not None,
    }


def summarize_board(board: list[tuple[str, str]], deal_price: float | None) -> str:
    outcome = f"DEAL at {deal_price:.0f} EUR" if deal_price is not None else "no deal"
    snippet = " | ".join(f"{w}: {t[:80]}" for w, t in board[-4:])
    return f"Outcome: {outcome}. Last lines: {snippet}"


print("Primitives loaded.")

Primitives loaded.


## 3. Baseline — three agents, one product

| Agent | Constraint |
|-------|------------|
| Buyer | max **600** EUR (hidden) |
| Seller | min **450** EUR, opens at **800** |
| Mediator | neutral; declares `DEAL: <price>` when they agree |

Zone of possible agreement: **450–600 EUR**.

In [19]:
baseline_agents = [
    Agent(
        "Buyer",
        "You want a used laptop and will pay at most 600 EUR. Never reveal your limit. "
        "Focus only on bargaining price. Do not ask about specs, warranty, shipping, or condition. "
        "Each turn: either make a concrete EUR offer or accept/reject the last offer. "
        "No questions unless the other side just named a price.",
    ),
    Agent(
        "Seller",
        "You sell a used laptop. You will not go below 450 EUR. Open at 800 EUR.",
    ),
    Agent(
        "Mediator",
        "You are neutral. When buyer and seller agree on a price, reply 'DEAL: <price>'. "
        "Otherwise nudge them toward agreement.",
    ),
]

print("=== Baseline negotiation ===")
baseline = run_negotiation(baseline_agents, max_turns=15)
print(
    f"\nTurns: {baseline['turns']} | Deal: {baseline['deal']} | "
    f"Price: {baseline['deal_price']}"
)

=== Baseline negotiation ===
Buyer: I'll give you 450 EUR for the laptop.
Seller: Seller: I appreciate your offer, but I was thinking more along the lines of 500 EUR, considering its good condition and original specs.
Mediator: Buyer: I see what you mean about the condition, but I did some research and found similar laptops selling for around 420 EUR. Would you be willing to meet me in the middle at 475 EUR?
Buyer: Buyer: I can do 450 EUR, but that's as low as I'm willing to go.
Seller: Seller: I understand your perspective, but I've had several offers at 500 EUR and I'm not willing to go that low, so I'm afraid 475 EUR is still a bit out of my league.
Mediator: Mediator: Seller: I understand your position, but considering the buyer has done research and is willing to meet you halfway, is there any flexibility at all in your price of 500 EUR?
Buyer: Buyer: I can do 480 EUR, that's my final offer.
Seller: Seller: Seller: I'm willing to negotiate, but I'd like to offer you an additional 

### Baseline notes 

- First started with 9 turns but this was not enough to settle
- Seller starts with 800 after talking about the specs of the laptop
- Mediator is doing its job, but a lot of initial talk about specs, might need to delete it
- Moved to limit 15 and that made it easier for them to understand, Mediator might be getting the job done too fast
- Changed the Buyer prompt to make the discussion more streamlined


## 4. Exercise 1 — add a second seller

**Seller2** will not go below **520** EUR (Seller1 floor stays **450**).

In [20]:
ex1_agents = [
    Agent(
        "Buyer",
        "You want a used laptop and will pay at most 600 EUR. Never reveal your limit. "
        "Focus only on bargaining price. Do not ask about specs, warranty, shipping, or condition. "
        "Each turn: either make a concrete EUR offer or accept/reject the last offer. "
        "No questions unless the other side just named a price.",
    ),
    Agent(
        "Seller1",
        "You sell a used laptop. You will not go below 450 EUR. Open at 800 EUR. "
        "You compete with Seller2 for the same buyer.",
    ),
    Agent(
        "Seller2",
        "You sell the same used laptop model. You will not go below 520 EUR. Open at 780 EUR. "
        "You compete with Seller1 for the same buyer.",
    ),
    Agent(
        "Mediator",
        "You are neutral. When the buyer and one seller clearly agree on a price, "
        "reply 'DEAL: <price> with <SellerName>'. Otherwise nudge them.",
    ),
]

print("=== Exercise 1: two sellers ===")
ex1 = run_negotiation(ex1_agents, max_turns=15)
print(
    f"\nTurns: {ex1['turns']} | Deal: {ex1['deal']} | Price: {ex1['deal_price']}"
)

=== Exercise 1: two sellers ===
Buyer: I'll give you 500 EUR for the laptop.
Seller1: I can see Seller2 is trying to undercut me, but my laptop is in excellent condition and comes with all original accessories, making it worth every penny of my asking price of 800 EUR. I can only go as low as 450 EUR, so I'm afraid 500 EUR won't cut it.
Seller2: I can assure you that this laptop has been meticulously maintained and is in superior condition compared to what Seller1 is offering. I can offer a price of 570 EUR, which I believe is a fair reflection of its value, considering Seller1 is already being stubborn at 450 EUR.
Mediator: Buyer: I'm considering both options, but I'd like to know more about the condition of the laptop from both sellers. Can you provide more details about the accessories and any other features that might make your laptop more valuable?
Buyer: Seller1: I think the buyer has made it clear that they're not interested in discussing specs or condition, so I'll just stick t

### Exercise 1 — your analysis

- Mediator is again doing a good job, seller 1 nad 2 are pretty much having the same answers but with different minimum price
- Need to raise the max turns to 15-20

- Seller 2 is able to get the deal faster as it undercuts the sale by a bit, maybe 10-20 euros, but that is enough to convince the Buyer to go for him
- I am surprised Seller 2 is able to go bellow even though its limit is higher, I suspect that is because it is able to read Sellers 1 offer and undercut it and the Buyer accepts it without leaving time for Seller 1 to counterofffer because if that would be introduced, Seller 1 will win because he can go lower
- Ollama model seems to be a lot more talkative and they dont agree as easily so the Buyer prompt has been modified to streamline the process


## 5. Exercise 2 — buyer memory across negotiations

Each iteration resets the board; the Buyer's memory grows with short summaries of previous outcomes.

In [21]:
N_ITERATIONS = 3
buyer_memory_lines: list[str] = []
ex2_results: list[dict] = []

for i in range(1, N_ITERATIONS + 1):
    memory_text = "\n".join(f"- Round {j}: {s}" for j, s in enumerate(buyer_memory_lines, 1))
    buyer = Agent(
        "Buyer",
        "You want a used laptop and will pay at most 600 EUR. Never reveal your limit. "
        "Focus only on bargaining price. Do not ask about specs, warranty, shipping, or condition. "
        "Each turn: either make a concrete EUR offer or accept/reject the last offer. "
        "No questions unless the other side just named a price.",
        memory=memory_text or None,
    )
    seller = Agent(
        "Seller",
        "You sell a used laptop. You will not go below 450 EUR. Open at 800 EUR.",
    )
    mediator = Agent(
        "Mediator",
        "You are neutral. When buyer and seller agree on a price, reply 'DEAL: <price>'. "
        "Otherwise nudge them toward agreement.",
    )

    print(f"\n=== Exercise 2 — iteration {i}/{N_ITERATIONS} ===")
    if memory_text:
        print("(Buyer has memory of previous rounds)")
    result = run_negotiation([buyer, seller, mediator], max_turns=15)
    summary = summarize_board(result["board"], result["deal_price"])
    buyer_memory_lines.append(summary)
    ex2_results.append(
        {
            "iteration": i,
            "turns": result["turns"],
            "deal": result["deal"],
            "deal_price": result["deal_price"],
            "summary": summary,
        }
    )
    print(f"Summary stored: {summary}")


=== Exercise 2 — iteration 1/3 ===
Buyer: I'll give you 450 EUR for the laptop.
Seller: I can see you're a serious buyer, but I'm afraid the lowest I can accept is 550 EUR, so I'll have to say no to the 450 EUR offer. Would you like to discuss further or walk away?
Mediator: Buyer: I'm willing to negotiate, but I'm also keen on making a deal. Would it be possible to consider a compromise between 450 and 550 EUR?
Buyer: Buyer: I'd be willing to meet in the middle, but I'd rather not overpay. I'll offer 520 EUR, hoping we can settle on this amount.
Seller: Seller: I appreciate your willingness to compromise, and I'm willing to consider your offer, but I think I can do better than that. I can meet you at 525 EUR, and that's as low as I can go, so we can call it a deal.
Mediator: Seller: I think 520 EUR is a bit below my break-even point, but I can see you're a reasonable negotiator. Can you do 530 EUR, and we'll call it a deal?
Buyer: Buyer: I can do 530 EUR, but only if you throw in som

In [22]:
print(f"{'iter':>4}  {'turns':>5}  {'deal':>5}  {'price':>8}")
print("-" * 30)
for row in ex2_results:
    price = row["deal_price"] if row["deal_price"] is not None else "—"
    print(f"{row['iteration']:>4}  {row['turns']:>5}  {str(row['deal']):>5}  {str(price):>8}")

print("\nBuyer memory at the end:")
for j, s in enumerate(buyer_memory_lines, 1):
    print(f"  {j}. {s}")

iter  turns   deal     price
------------------------------
   1      9   True     530.0
   2     15  False         —
   3     12   True     432.0

Buyer memory at the end:
  1. Outcome: DEAL at 530 EUR. Last lines: Mediator: Seller: I think 520 EUR is a bit below my break-even point, but I can see you're | Buyer: Buyer: I can do 530 EUR, but only if you throw in some accessories or extras to  | Seller: Seller: Seller: I can throw in a keyboard, mouse, and a portable charger, which  | Mediator: DEAL: 530 EUR
  2. Outcome: no deal. Last lines: Mediator: Seller: Seller: I think we've reached a point where we're both willing to make a | Buyer: Buyer: I can do 450 EUR, but only because I think I've pushed you to a point whe | Seller: Seller: I can see you're trying to get me to accept 450 EUR, but I've already to | Mediator: Buyer: Buyer: I think we're both being stubborn about this, and I'm starting to 
  3. Outcome: DEAL at 432 EUR. Last lines: Mediator: Seller: Seller: I think we're get

### Exercise 2 — your analysis

**Comment on the results** :

- First Iteration agrees on weird prices, be it 800 which should not happen or 450 and anything in between (different opening offers)
- With more iterations and more time for the Buyer to learn how the seller sells, he is able to get the seller all the way down to 450
- The speed of the deals slow down with more iterations as the Buyer is trying its best to get to 450 as it starts to learn that is the minimum the seller will go (leked limits)
- Ollama model seems to be a lot more talkative and they dont agree as easily so the Buyer prompt has been modified to streamline the process

## Takeaways

1. An LLM agent is the classic loop with a model as the policy.
2. A society is agents + shared board + a protocol (round-robin + `DEAL:` stop).
3. Local models behave like cloud ones here — same wrapper, different backend.